# Git 第1周:内部原理 — Git 是怎么存数据的

> **学习目标**:理解 .git 目录的结构,掌握 blob/tree/commit/ref 四种对象,摆脱"Git 是黑盒"的感觉

---

## Day 1:快照,不是差异

### 其他 VCS vs Git 的存储模型

| 对比 | Delta-based(SVN等) | Snapshot-based(Git) |
|------|---------------------|----------------------|
| 存储方式 | 每次提交存 diff | 每次提交存完整文件树 |
| 查看历史 | 需要从第一个版本逐步应用 diff | 直接取出该版本的完整快照 |
| 分支切换 | 慢(需要重建文件) | 极快(只改指针) |
| 数据完整性 | 依赖服务器 | 每个对象都有 SHA-1 校验 |

### 为什么 Git 选快照模型？

1. **分支切换极快** — 不需要重建文件,只改 HEAD 指针
2. **离线操作** — 完整仓库在本地,不依赖网络
3. **数据完整性** — SHA-1 哈希保证内容不被篡改
4. **压缩策略** — 定期 `git gc` 会把松散对象打包 + delta 压缩,兼顾速度和空间

### 练习:模拟 Git 对象存储

下面的代码模拟了 `git init` 和提交的过程,观察 SHA-1 的计算方式。

In [ ]:
import os, hashlib, tempfile, shutil

tmp = tempfile.mkdtemp(prefix='git-demo-')
git_dir = os.path.join(tmp, '.git')
objects_dir = os.path.join(git_dir, 'objects')
os.makedirs(objects_dir)

def git_hash_object(content):
    """计算 Git blob 的 SHA-1:'blob <size>\0<content>'"""
    header = f"blob {len(content)}\0"
    store = header.encode() + (content if isinstance(content, bytes) else content.encode())
    return hashlib.sha1(store).hexdigest()

# 模拟创建两个版本的文件
v1 = git_hash_object("Hello Git!")
v2 = git_hash_object("Hello Git! This is version 2.")
v1_same = git_hash_object("Hello Git!")

print(f"v1 内容的 SHA-1: {v1}")
print(f"v2 内容的 SHA-1: {v2}")
print(f"相同内容的 SHA-1:  {v1_same}")
print(f"SHA-1 是否相同: {v1 == v1_same}")
print()
print("关键洞察:Git 中,内容相同 → SHA-1 相同 → 只存一份!")
print("这就是为什么 Git 不会重复存储相同内容的文件。")

shutil.rmtree(tmp)

## Day 2:三种对象 — blob / tree / commit

Git 的 `.git/objects/` 里存了四种对象。核心是前三种:

```
commit  ←── 一次提交:指向 tree + 父 commit + 元信息
  └── tree  ←── 目录结构:文件名 → blob 的映射
       └── blob  ←── 文件内容(不含文件名!)
```

### 用底层命令看一眼真实对象

In [ ]:
import subprocess, os

os.chdir('/home/oa/utils/devops-study')

def run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("=" * 50)
print("1. 最近的 commit:")
print("=" * 50)
print(run("git log --oneline -3"))

print()
print("=" * 50)
print("2. HEAD commit 对象内容(git cat-file -p HEAD):")
print("=" * 50)
print(run("git cat-file -p HEAD"))

print()
print("=" * 50)
print("3. HEAD 指向的 tree 对象:")
print("=" * 50)
print(run("git cat-file -p HEAD^{tree}"))

print()
print("=" * 50)
print("4. tree 内容列表(git ls-tree HEAD):")
print("=" * 50)
print(run("git ls-tree HEAD"))

### 对象关系示意图

```
commit: a1b2c3...
├── tree: d4e5f6...
│   ├── blob: 789abc...  README.md
│   ├── blob: def012...  docker_learning_plan.md
│   ├── tree: 345678...  docs/
│   │   └── ...
│   └── ...
├── parent: x9y8z7...  (上一个 commit)
├── author: GAOSHIQING
└── message: "w"
```

关键理解:
- **blob 不存文件名**!文件名存在 tree 里。
- **tree 不存 commit message**!元信息存在 commit 里。
- 三者分离设计让 Git 可以高效去重和复用。

## Day 3:暂存区(index)的本质

### 三个区域的关系

```
工作目录 (Working Directory)    →  暂存区 (Index/Staging)   →  版本库 (Repository)
    你编辑的文件                    git add 后暂存               git commit 后永久保存
```

`git status` 实际上在比较这三个区域:
- 工作区 vs 暂存区 → `git diff`(红色,unstaged changes)
- 暂存区 vs HEAD → `git diff --cached`(绿色,staged changes)
- 工作区 vs HEAD → `git diff HEAD`(所有未提交的改动)

In [ ]:
# 模拟 git status 的三个比较

head = {"README.md": "v1", "plan.md": "v1"}       # HEAD commit
index = {"README.md": "v1", "plan.md": "v2"}       # 暂存区 (git add 后)
worktree = {"README.md": "v3", "plan.md": "v2"}   # 工作区 (编辑后)

print("当前状态:")
print(f"  HEAD (上次 commit):   {head}")
print(f"  Index (暂存区):        {index}")
print(f"  Worktree (工作目录):   {worktree}")

print()
print("=== git status 分析 ===")

# 工作区 vs 暂存区 (git diff)
print("\nChanges not staged (工作区 vs 暂存区 - 红色):")
for f in set(list(worktree.keys()) + list(index.keys())):
    if worktree.get(f) != index.get(f):
        print(f"  modified: {f}")

# 暂存区 vs HEAD (git diff --cached)
print("\nChanges to be committed (暂存区 vs HEAD - 绿色):")
for f in set(list(index.keys()) + list(head.keys())):
    if index.get(f) != head.get(f):
        print(f"  modified: {f}")

print()
print("关键要点:")
print("- git diff         → 比较 worktree vs index(还没暂存的修改)")
print("- git diff --cached → 比较 index vs HEAD(已暂存、待提交的修改)")
print("- git diff HEAD     → 比较 worktree vs HEAD(所有的修改)")

## Day 4:引用(refs)— 分支名的真相

### refs 目录结构

```
.git/
├── HEAD              → ref: refs/heads/main  (指向当前分支)
├── refs/
│   ├── heads/        → 本地分支
│   │   ├── main      → 40位 SHA
│   │   └── feature   → 40位 SHA
│   ├── remotes/      → 远程跟踪分支
│   │   └── origin/
│   │       └── main  → 40位 SHA
│   └── tags/         → 标签
│       └── v1.0      → 40位 SHA
```

**分支 = 一个指向 commit 的指针**,就是一个 40 字节的文件。

In [ ]:
import subprocess, os

os.chdir('/home/oa/utils/devops-study')

def run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("当前 .git/refs/ 目录结构:")
print(run("find .git/refs -type f | head -20"))

print()
print("HEAD 内容:")
print(run("cat .git/HEAD"))

print()
print("main 分支指向的 SHA:")
print(run("cat .git/refs/heads/main"))

print()
print("所有本地分支(就是 refs/heads/ 里的文件):")
print(run("git branch"))

print()
print("关键洞察:")
print("- 创建一个分支,就是在 .git/refs/heads/ 下新建一个文件")
print("- 文件内容就是一个 40 位的 SHA-1")
print("- 所以创建分支是 O(1) 操作,几乎没有开销")

## Day 5:对象存储 — 松散对象与打包

### 两种存储形式

| 形式 | 路径 | 说明 |
|------|------|------|
| 松散对象(loose) | `.git/objects/xx/xxxxxx...` | 每个对象一个文件 |
| 打包文件(pack) | `.git/objects/pack/*.pack` + `*.idx` | 多个对象打包 + delta 压缩 |

Git 的策略:先以松散对象存储(快),定期 `git gc` 把松散对象打包(省空间)。

In [ ]:
import subprocess, os

os.chdir('/home/oa/utils/devops-study')

def run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("对象统计:")
print(run("git count-objects -vH"))

print()
print("pack 文件:")
print(run("ls -lh .git/objects/pack/"))

print()
print("松散对象数量:")
print(run("find .git/objects -type f -not -path '*/pack/*' | wc -l"))

print()
print("触发垃圾回收:")
print("  git gc --aggressive  # 打包所有松散对象")
print("  git gc --auto        # Git 自动判断是否需要 gc")

## Day 6:reset / restore — 理解本质才能不误操作

### reset 的三个模式

| 模式 | HEAD | Index | Worktree | 用途 |
|------|------|-------|-----------|------|
| `--soft` | 移动 | 不变 | 不变 | 撤销 commit 但保留所有修改(重新 commit 用) |
| `--mixed` (默认) | 移动 | 重置 | 不变 | "撤销 commit 和 add,但保留文件修改" |
| `--hard` | 移动 | 重置 | 重置 | 完全回到某个版本(危险!) |

In [ ]:
# 模拟 reset 三种模式

class GitSim:
    def __init__(self):
        self.head = "commit-3"
        self.commits = {"commit-1": {"f": "v1"}, "commit-2": {"f": "v2"}, "commit-3": {"f": "v3"}}
        self.index = {"f": "v3"}
        self.wt = {"f": "v3"}

    def reset(self, target, mode):
        self.head = target
        if mode == "hard":
            self.index = dict(self.commits[target])
            self.wt = dict(self.commits[target])
        elif mode == "mixed":
            self.index = dict(self.commits[target])
        print(f"git reset --{mode} {target}")
        print(f"  HEAD={self.head}, index={self.index}, wt={self.wt}")

gs1 = GitSim()
gs1.reset("commit-2", "soft")
print("  → 修改仍在暂存区,可以直接 git commit")

gs2 = GitSim()
gs2.reset("commit-2", "mixed")
print("  → 修改仍在工作区,需要重新 git add + git commit")

gs3 = GitSim()
gs3.reset("commit-2", "hard")
print("  → 所有修改丢失!只能通过 reflog 找回")

## Day 7:第1周综合练习

In [ ]:
print("=" * 60)
print("第1周综合练习:从零构建 Git 的对象模型")
print("=" * 60)

import hashlib, json

class TinyGit:
    def __init__(self):
        self.objects = {}
        self.refs = {"refs/heads/main": None}
        self.index = {}

    def hash_object(self, data, obj_type="blob"):
        header = f"{obj_type} {len(data)}\0"
        store = header.encode() + (data if isinstance(data, bytes) else data.encode())
        return hashlib.sha1(store).hexdigest()

    def add(self, filename, content):
        sha = self.hash_object(content)
        self.objects[sha] = ("blob", content)
        self.index[filename] = sha
        print(f"  git add {filename}  → blob {sha[:8]}")

    def commit(self, message):
        tree_content = json.dumps(self.index, sort_keys=True)
        tree_sha = self.hash_object(tree_content, "tree")
        self.objects[tree_sha] = ("tree", tree_content)

        parent = self.refs["refs/heads/main"]
        commit_data = json.dumps({"tree": tree_sha, "parent": parent, "message": message})
        commit_sha = self.hash_object(commit_data, "commit")
        self.objects[commit_sha] = ("commit", commit_data)
        self.refs["refs/heads/main"] = commit_sha
        print(f"  git commit -m '{message}' → commit {commit_sha[:8]}")

git = TinyGit()
print("\n1. 创建文件并提交:")
git.add("README.md", "# My Project")
git.commit("Initial commit")

print("\n2. 添加更多文件:")
git.add("main.py", "print('hello')")
git.commit("Add main.py")

print("\n3. 提交后的对象库:")
for sha, (typ, content) in git.objects.items():
    print(f"  {sha[:8]} ({typ})")

print(f"\n4. refs/heads/main → {git.refs['refs/heads/main'][:8]}")
print()
print("=" * 60)
print("第1周核心收获:")
print("1. Git 是快照模型 —— 每次 commit 存完整状态,不是 diff")
print("2. blob 存内容、tree 存目录、commit 存元信息 —— 三者分离")
print("3. 分支就是一个指向 commit 的指针文件")
print("4. git add 创建 blob 更新 index;git commit 创建 tree+commit 更新 ref")
print("=" * 60)